<a href="https://colab.research.google.com/github/Sabari19-adda/A-Selective-Feature-Layer-and-Soft-Label-Fused-Knowledge-Distillation-System-/blob/main/Student_WKD_ResNet101V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import time
import numpy as np
import tensorflow as tf
from PIL import Image
import os
from collections import defaultdict

# ===== STEP 1: Class Labels =====
class_labels = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

# ===== STEP 2: Load the Keras model =====
# Update this path to your .h5 or SavedModel directory
model = tf.keras.models.load_model("/content/drive/MyDrive/Patent_models/Student-WOKD-Run2.keras")

# ===== STEP 3: Get input shape =====
input_shape = model.input_shape
height, width = input_shape[1], input_shape[2]

# ===== STEP 4: Preprocess function =====
def preprocess_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img = img.resize((width, height))
    img = np.array(img).astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    return img

# ===== STEP 5: Define image paths (5 from each class) =====
sample_dir = "/content/drive/MyDrive/Patent_models/RasExp/sampleimgs_new"
image_paths = []

for label in class_labels:
    class_path = os.path.join(sample_dir, label)
    images = os.listdir(class_path)[:5]
    for img_name in images:
        image_paths.append((os.path.join(class_path, img_name), label))

# ===== STEP 6: Run inference on all images =====
class_specific_times = defaultdict(list)

print("----- Inference Results (Keras Model) -----")

for image_path, true_class in image_paths:
    input_data = preprocess_image(image_path)

    start_time = time.perf_counter()
    output_data = model.predict(input_data, verbose=0)
    end_time = time.perf_counter()

    predicted_index = np.argmax(output_data)
    predicted_label = class_labels[predicted_index]
    confidence = output_data[0][predicted_index]

    inference_time_ms = (end_time - start_time) * 1000

    class_specific_times[true_class].append(inference_time_ms)

    print(f"True Class     : {true_class}")
    print(f"Predicted Class: {predicted_label}")
    print(f"Confidence     : {confidence * 100:.2f}%")
    print(f"Inference Time : {inference_time_ms:.2f} ms")
    print("-" * 40)

# ===== STEP 7: Print Mean Time for Each Class =====
print("\n📊 Mean Inference Time per Class:")
print("-" * 40)

all_times = []

for label in class_labels:
    times = class_specific_times[label]
    if times:
        mean_class_time = sum(times) / len(times)
        all_times.extend(times)
        print(f"✅ {label:<20}: {mean_class_time:.2f} ms")

avg_total_time = sum(all_times) / len(all_times)

print("-" * 40)
print(f"✅ Average Inference Time over {len(all_times)} images: {avg_total_time:.2f} ms")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
----- Inference Results (Keras Model) -----
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 99.38%
Inference Time : 1702.16 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 99.15%
Inference Time : 210.38 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 99.96%
Inference Time : 357.77 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 99.99%
Inference Time : 213.74 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 199.38 ms
----------------------------------------
True Class     : ModerateDemented
Predicted Class: ModerateDement

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model
import os
from collections import defaultdict

# ===== STEP 1: Register Custom Distiller Class =====
@tf.keras.utils.register_keras_serializable()
class Distiller(tf.keras.Model):
    def __init__(self, student, teacher, alpha=0.5, temperature=3.0):
        super().__init__()
        self.student = student
        self.teacher = teacher

    def call(self, x):
        return self.student(x)

# ===== STEP 2: Class Labels =====
class_labels = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

# ===== STEP 3: Load & Extract Student =====
model_path = "/content/drive/MyDrive/Patent_models/Student-WKD-ResNet101V2-Run1+2.keras"

if not os.path.exists(model_path):
    print(f"❌ File not found at: {model_path}")
else:
    tf.keras.backend.clear_session()

    # Load the full Distiller/Sequential object
    full_model = load_model(model_path, compile=False)

    # 🔥 THE FIX: Extract the core student model
    # If it's a Distiller, take .student. If it's a Sequential wrapper, take layer 0.
    if hasattr(full_model, 'student'):
        inference_model = full_model.student
        print("✅ Extracted Student from Distiller.")
    elif isinstance(full_model, tf.keras.Sequential):
        inference_model = full_model.layers[0]
        print("✅ Extracted Student from Sequential wrapper.")
    else:
        inference_model = full_model
        print("✅ Using model as standard functional network.")

    # ===== STEP 4: Preprocess function =====
    img_size = (224, 224)

    def preprocess_image(image_path):
        img = load_img(image_path, target_size=img_size)
        img_array = img_to_array(img) / 255.0
        # Ensure it's a 4D float32 tensor (Batch, H, W, C)
        img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)
        return img_tensor[tf.newaxis, ...]

    # ===== STEP 5: Define image paths =====
    sample_dir = "/content/drive/MyDrive/Patent_models/RasExp/sampleimgs_new"
    image_paths = []
    for label in class_labels:
        class_path = os.path.join(sample_dir, label)
        if os.path.exists(class_path):
            images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:5]
            for img_name in images:
                image_paths.append((os.path.join(class_path, img_name), label))

    # ===== STEP 6: Run inference =====
    class_specific_times = defaultdict(list)
    all_times = []

    print(f"\n----- Inference Results (Direct Student Call) -----\n")

    for image_path, true_class in image_paths:
        input_data = preprocess_image(image_path)

        # Timing the direct call to the extracted student model
        # This bypasses the problematic Softmax wrapper/Distiller call
        start_time = time.perf_counter()
        output_data = inference_model(input_data, training=False)
        end_time = time.perf_counter()

        output_numpy = output_data.numpy()
        predicted_index = np.argmax(output_numpy)
        predicted_label = class_labels[predicted_index]
        confidence = output_numpy[0][predicted_index]

        inference_time_ms = (end_time - start_time) * 1000
        class_specific_times[true_class].append(inference_time_ms)
        all_times.append(inference_time_ms)

        print(f"True Class     : {true_class}")
        print(f"Predicted Class: {predicted_label}")
        print(f"Confidence     : {confidence * 100:.2f}%")
        print(f"Inference Time : {inference_time_ms:.2f} ms")
        print("-" * 40)

    # ===== STEP 7: Final Metrics Report =====
    print("\n📊 Final Performance Report:")
    print("-" * 40)
    for label in class_labels:
        if class_specific_times[label]:
            mean_time = sum(class_specific_times[label]) / len(class_specific_times[label])
            print(f"✅ {label:<20}: {mean_time:.2f} ms")

    if all_times:
        print("-" * 40)
        print(f"✅ Overall Average Inference Time: {np.mean(all_times):.2f} ms")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 84 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


✅ Extracted Student from Sequential wrapper.

----- Inference Results (Direct Student Call) -----

True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 127.84 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 99.99%
Inference Time : 71.44 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 72.30 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 64.15 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 69.50 ms
----------------------------------------
True Class     : ModerateDemented
Predicted Class: ModerateDemented
Confidence     : 100.00%
Inference Time : 96.12 ms
----------------------

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model
import os
from collections import defaultdict

# ===== STEP 1: Register Custom Distiller Class =====
@tf.keras.utils.register_keras_serializable()
class Distiller(tf.keras.Model):
    def __init__(self, student, teacher, alpha=0.5, temperature=3.0):
        super().__init__()
        self.student = student
        self.teacher = teacher

    def call(self, x):
        return self.student(x)

# ===== STEP 2: Class Labels =====
class_labels = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

# ===== STEP 3: Load & Extract Student =====
model_path = "/content/drive/MyDrive/Patent_models/Student-WKD-InceptionV3-Run2.keras"

if not os.path.exists(model_path):
    print(f"❌ File not found at: {model_path}")
else:
    tf.keras.backend.clear_session()

    # Load the full Distiller/Sequential object
    full_model = load_model(model_path, compile=False)

    # 🔥 THE FIX: Extract the core student model
    # If it's a Distiller, take .student. If it's a Sequential wrapper, take layer 0.
    if hasattr(full_model, 'student'):
        inference_model = full_model.student
        print("✅ Extracted Student from Distiller.")
    elif isinstance(full_model, tf.keras.Sequential):
        inference_model = full_model.layers[0]
        print("✅ Extracted Student from Sequential wrapper.")
    else:
        inference_model = full_model
        print("✅ Using model as standard functional network.")

    # ===== STEP 4: Preprocess function =====
    img_size = (224, 224)

    def preprocess_image(image_path):
        img = load_img(image_path, target_size=img_size)
        img_array = img_to_array(img) / 255.0
        # Ensure it's a 4D float32 tensor (Batch, H, W, C)
        img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)
        return img_tensor[tf.newaxis, ...]

    # ===== STEP 5: Define image paths =====
    sample_dir = "/content/drive/MyDrive/Patent_models/RasExp/sampleimgs_new"
    image_paths = []
    for label in class_labels:
        class_path = os.path.join(sample_dir, label)
        if os.path.exists(class_path):
            images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:5]
            for img_name in images:
                image_paths.append((os.path.join(class_path, img_name), label))

    # ===== STEP 6: Run inference =====
    class_specific_times = defaultdict(list)
    all_times = []

    print(f"\n----- Inference Results (Direct Student Call) -----\n")

    for image_path, true_class in image_paths:
        input_data = preprocess_image(image_path)

        # Timing the direct call to the extracted student model
        # This bypasses the problematic Softmax wrapper/Distiller call
        start_time = time.perf_counter()
        output_data = inference_model(input_data, training=False)
        end_time = time.perf_counter()

        output_numpy = output_data.numpy()
        predicted_index = np.argmax(output_numpy)
        predicted_label = class_labels[predicted_index]
        confidence = output_numpy[0][predicted_index]

        inference_time_ms = (end_time - start_time) * 1000
        class_specific_times[true_class].append(inference_time_ms)
        all_times.append(inference_time_ms)

        print(f"True Class     : {true_class}")
        print(f"Predicted Class: {predicted_label}")
        print(f"Confidence     : {confidence * 100:.2f}%")
        print(f"Inference Time : {inference_time_ms:.2f} ms")
        print("-" * 40)

    # ===== STEP 7: Final Metrics Report =====
    print("\n📊 Final Performance Report:")
    print("-" * 40)
    for label in class_labels:
        if class_specific_times[label]:
            mean_time = sum(class_specific_times[label]) / len(class_specific_times[label])
            print(f"✅ {label:<20}: {mean_time:.2f} ms")

    if all_times:
        print("-" * 40)
        print(f"✅ Overall Average Inference Time: {np.mean(all_times):.2f} ms")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Extracted Student from Sequential wrapper.

----- Inference Results (Direct Student Call) -----

True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 156.98 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 128.62 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 114.15 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 99.98%
Inference Time : 132.07 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 111.85 ms
----------------------------------------
True Cla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import time
import numpy as np
import tensorflow as tf
from PIL import Image
import os
from collections import defaultdict

# ===== STEP 1: Class Labels =====
class_labels = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

# ===== STEP 2: Load the Keras model =====
# Update this path to your .h5 or SavedModel directory
model = tf.keras.models.load_model("/content/drive/MyDrive/Patent_models/LR/Incep_ADAM_0.001_Run2_final.keras")

# ===== STEP 3: Get input shape =====
input_shape = model.input_shape
height, width = input_shape[1], input_shape[2]

# ===== STEP 4: Preprocess function =====
def preprocess_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img = img.resize((width, height))
    img = np.array(img).astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    return img

# ===== STEP 5: Define image paths (5 from each class) =====
sample_dir = "/content/drive/MyDrive/Patent_models/RasExp/sampleimgs_new"
image_paths = []

for label in class_labels:
    class_path = os.path.join(sample_dir, label)
    images = os.listdir(class_path)[:5]
    for img_name in images:
        image_paths.append((os.path.join(class_path, img_name), label))

# ===== STEP 6: Run inference on all images =====
class_specific_times = defaultdict(list)

print("----- Inference Results (Keras Model) -----")

for image_path, true_class in image_paths:
    input_data = preprocess_image(image_path)

    start_time = time.perf_counter()
    output_data = model.predict(input_data, verbose=0)
    end_time = time.perf_counter()

    predicted_index = np.argmax(output_data)
    predicted_label = class_labels[predicted_index]
    confidence = output_data[0][predicted_index]

    inference_time_ms = (end_time - start_time) * 1000

    class_specific_times[true_class].append(inference_time_ms)

    print(f"True Class     : {true_class}")
    print(f"Predicted Class: {predicted_label}")
    print(f"Confidence     : {confidence * 100:.2f}%")
    print(f"Inference Time : {inference_time_ms:.2f} ms")
    print("-" * 40)

# ===== STEP 7: Print Mean Time for Each Class =====
print("\n📊 Mean Inference Time per Class:")
print("-" * 40)

all_times = []

for label in class_labels:
    times = class_specific_times[label]
    if times:
        mean_class_time = sum(times) / len(times)
        all_times.extend(times)
        print(f"✅ {label:<20}: {mean_class_time:.2f} ms")

avg_total_time = sum(all_times) / len(all_times)

print("-" * 40)
print(f"✅ Average Inference Time over {len(all_times)} images: {avg_total_time:.2f} ms")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
----- Inference Results (Keras Model) -----
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 1361.25 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 114.70 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 93.50 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 93.64 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 89.75 ms
----------------------------------------
True Class     : ModerateDemented
Predicted Class: ModerateDemen

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import time
import numpy as np
import tensorflow as tf
from PIL import Image
import os
from collections import defaultdict

# ===== STEP 1: Class Labels =====
class_labels = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

# ===== STEP 2: Load the Keras model =====
# Update this path to your .h5 or SavedModel directory
model = tf.keras.models.load_model("/content/drive/MyDrive/Patent_models/LR/ResNet101V2_ADAM_0.001_final_student_Run2.keras")

# ===== STEP 3: Get input shape =====
input_shape = model.input_shape
height, width = input_shape[1], input_shape[2]

# ===== STEP 4: Preprocess function =====
def preprocess_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img = img.resize((width, height))
    img = np.array(img).astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    return img

# ===== STEP 5: Define image paths (5 from each class) =====
sample_dir = "/content/drive/MyDrive/Patent_models/RasExp/sampleimgs_new"
image_paths = []

for label in class_labels:
    class_path = os.path.join(sample_dir, label)
    images = os.listdir(class_path)[:5]
    for img_name in images:
        image_paths.append((os.path.join(class_path, img_name), label))

# ===== STEP 6: Run inference on all images =====
class_specific_times = defaultdict(list)

print("----- Inference Results (Keras Model) -----")

for image_path, true_class in image_paths:
    input_data = preprocess_image(image_path)

    start_time = time.perf_counter()
    output_data = model.predict(input_data, verbose=0)
    end_time = time.perf_counter()

    predicted_index = np.argmax(output_data)
    predicted_label = class_labels[predicted_index]
    confidence = output_data[0][predicted_index]

    inference_time_ms = (end_time - start_time) * 1000

    class_specific_times[true_class].append(inference_time_ms)

    print(f"True Class     : {true_class}")
    print(f"Predicted Class: {predicted_label}")
    print(f"Confidence     : {confidence * 100:.2f}%")
    print(f"Inference Time : {inference_time_ms:.2f} ms")
    print("-" * 40)

# ===== STEP 7: Print Mean Time for Each Class =====
print("\n📊 Mean Inference Time per Class:")
print("-" * 40)

all_times = []

for label in class_labels:
    times = class_specific_times[label]
    if times:
        mean_class_time = sum(times) / len(times)
        all_times.extend(times)
        print(f"✅ {label:<20}: {mean_class_time:.2f} ms")

avg_total_time = sum(all_times) / len(all_times)

print("-" * 40)
print(f"✅ Average Inference Time over {len(all_times)} images: {avg_total_time:.2f} ms")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
----- Inference Results (Keras Model) -----
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 385.99 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 104.91 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 111.15 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 107.45 ms
----------------------------------------
True Class     : MildDemented
Predicted Class: MildDemented
Confidence     : 100.00%
Inference Time : 105.91 ms
----------------------------------------
True Class     : ModerateDemented
Predicted Class: ModerateDem